In [1]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.models.network import DiffusionAttn
from src.models.diffusion import GaussianDiffusion
from src.train.trainer import setup_optimizer, DiffusionTrainer
from src.utils.seed import set_seed

/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF


/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_seed(42)

In [3]:
# Global hyperpar
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.0008
WEIGHT_DECAY = 0.01
TIMESTEPS = 1000 

TEST_INHIBITOR = "2-mercaptobenzimidazole" 

NUM_CYCLE = [1, 2, 3, 4]
save_dir = project_root / "experiments" / "run_01"
SAVE_DIR = str(save_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

[*] Device: cuda


In [4]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=TEST_INHIBITOR, 
    norm_feat=True, 
    use_wavelet=False
)

train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)

val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)

In [5]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

Size Train: 2684 samples
Size Val: 776 samples


In [6]:
num_desc_features = train_dataset[0]["features"].shape[0]

net = DiffusionAttn(
        in_channels=1, 
        desc_features=num_desc_features, 
        base_channels=64
    )
    
diffusion = GaussianDiffusion(model=net, timesteps=TIMESTEPS)

optimizer, scheduler = setup_optimizer(
    model=net, 
    lr=LR, 
    weight_decay=WEIGHT_DECAY, 
    epochs=EPOCHS
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
trainer = DiffusionTrainer(
        diffusion_model=diffusion,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        save_dir=SAVE_DIR, 
        vol_scaler=pipe.vol_scaler,
        cur_scaler=pipe.cur_scaler
    )

print("\n" + "="*40)
print("Start")
print("="*40)
trainer.fit(epochs=EPOCHS)


Start
Teaching on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:01<00:00,  6.67it/s, val_loss=0.0182]


Epoch 1 | Train Loss: 0.0369 | Val Loss: 0.0167 | LR: 0.000800
            | Noise Loss: 0.0369 | Bounds Loss: 0.0004 | TV Loss: 0.0671
Saved best model (Val Loss: 0.0167)


Epoch 2 [Val]: 100%|██████████| 13/13 [00:01<00:00,  7.04it/s, val_loss=0.0088]


Epoch 2 | Train Loss: 0.0134 | Val Loss: 0.0097 | LR: 0.000799
            | Noise Loss: 0.0134 | Bounds Loss: 0.0000 | TV Loss: 0.0253
Saved best model (Val Loss: 0.0097)


...

Epoch 38 [Val]: 100%|██████████| 13/13 [00:02<00:00,  6.27it/s, val_loss=0.0098]


Epoch 38 | Train Loss: 0.0040 | Val Loss: 0.0136 | LR: 0.000547
            | Noise Loss: 0.0040 | Bounds Loss: 0.0000 | TV Loss: 0.0041


Epoch 39 [Val]: 100%|██████████| 13/13 [00:02<00:00,  6.33it/s, val_loss=0.0051]


Epoch 39 | Train Loss: 0.0040 | Val Loss: 0.0070 | LR: 0.000535
            | Noise Loss: 0.0040 | Bounds Loss: 0.0000 | TV Loss: 0.0043


Epoch 40 [Train]:  63%|██████▎   | 26/41 [00:13<00:07,  1.88it/s, loss=0.0032]
